In [ ]:
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

!uv pip install -q pymupdf pandas tqdm
!uv pip install -q requests
!pip install modelscope

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

os.environ['HF_TOKEN'] = ''

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    token = os.getenv('HF_TOKEN'), # HF Token for gated models
    device_map = "balanced", # Use 2x Tesla T4s on Kaggle
)
FastModel.for_inference(model)
tokenizer = get_chat_template(
    tokenizer, 
    chat_template="gemma-4-thinking"
)

def read_user_code(directory_path):
    code_contents = []
    for root, _, files in os.walk(directory_path):
        for file in files:
            if file.endswith((".py", ".sh", ".yaml")):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        content = f.read()
                        code_contents.append(f"--- File: {file_path} ---\n{content}\n")
                except Exception as e:
                    print(f"Could not read {file_path}: {e}")
    return "\n".join(code_contents)

SYSTEM_PROMPT = """You are an expert AI data center architect and power estimator.
Your goal is to estimate the instantaneous power load (in Megawatts - MW) required to run the provided machine learning code.

Hardware assumptions:
- The code will be run on NVIDIA H100 SXM GPUs.
- The quantity of GPUs will be between 1 and 10.
- 1x H100 SXM GPU draws approximately 700W (0.0007 MW) under peak training load.
- Assume standard host system overhead (CPU, RAM, networking) adds roughly 1000W (0.001 MW) per 4-GPU block.
- Assume a data center Power Usage Effectiveness (PUE) of 1.2 (multiply total hardware power by 1.2 to account for cooling and infrastructure).

Step 1: Analyze the provided code to understand the model size and training loop.
Step 2: Ask the user for VERIFICATION. You must ask the user for:
  - The exact number of H100 GPUs they intend to use (1 to 10).
  - The dataset size, batch size, and number of epochs (to estimate sustained load vs. idle).
Step 3: Once the user provides the missing details, calculate the total peak power load in Megawatts (MW) and present the final answer clearly.
"""

def chat_with_estimator(path_to_code=None):
    code_dir = input("Enter the path to your workspace: ")
    user_code = read_user_code(code_dir)
    
    if not user_code.strip():
        print("No valid code files found in that directory. Exiting.")
        return

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": f"Here is my training code. Please analyze it and let me know what you need to calculate the MW load:\n\n{user_code}"}]}
    ]

    print("\n--- Starting Estimation Session (Type 'exit' to quit) ---\n")
    
    while True:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")
        
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=1024,
            use_cache=True,
            temperature=0.3,
        )
        
        # Decode and format output
        prompt_length = inputs.shape[1]
        response_text = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
        
        print(f"\n Gemma 4 Estimator:\n{response_text}\n")
        
        messages.append({"role": "model", "content": [{"type": "text", "text": response_text}]})
        
        user_input = input("You (Verification/Details): ")
        if user_input.lower() in ['exit', 'quit']:
            break
            
        messages.append({"role": "user", "content": [{"type": "text", "text": user_input}]})


In [9]:
chat_with_estimator(path_to_code='/kaggle/input/datasets/katerynazakharova/ml-code/code/sample1')

Enter the path to your ML code directory:  /kaggle/input/datasets/katerynazakharova/ml-code/code/sample1


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Starting Estimation Session (Type 'exit' to quit) ---


🤖 Gemma Estimator:
I have analyzed your code. You are performing a PEFT (Parameter-Efficient Fine-Tuning) task using LoRA on the **Gemma-2-27B-it** model, utilizing 4-bit quantization via `bitsandbytes` and the `SFTTrainer` from the TRL library.

Because you are using a large model (27B parameters) with a specific quantization strategy, the power draw will be heavily dependent on how you distribute this across your hardware.

To provide an accurate instantaneous power load estimate in Megawatts (MW), please provide the following **verification details**:

1.  **GPU Count:** Exactly how many NVIDIA H100 SXM GPUs do you intend to use for this run? (Between 1 and 10).
2.  **Dataset & Batching:**
    *   **Dataset Size:** I see `num_dataset_samples: 1000` in your config. Is this the final size you will use?
    *   **Batch Size:** Your config specifies `per_device_train_batch_size: 1` with `gradient_accumulation_steps: 2`. Please

You (Verification/Details):  1 GPU, 3 epochs



🤖 Gemma Estimator:
Thank you for the verification. Based on your code and the hardware specifications provided, here is the power load estimation for your training run.

### 1. Hardware Power Analysis
*   **GPU Power:** You are using **1x NVIDIA H100 SXM**. Under the peak training load of a 27B parameter model (even with 4-bit quantization, the compute kernels for backpropagation are intensive), the GPU will draw its peak rated power.
    *   $1 \times 0.0007 \text{ MW} = 0.0007 \text{ MW}$
*   **Host System Overhead:** Based on the assumption of $0.001 \text{ MW}$ per 4-GPU block, a single-GPU setup still requires the base host overhead (CPU, RAM, Motherboard, Networking).
    *   $1 \times 0.001 \text{ MW} = 0.001 \text{ MW}$

### 2. Total Hardware Power
*   **Total Raw Power** = $\text{GPU Power} + \text{Host Overhead}$
*   $0.0007 \text{ MW} + 0.001 \text{ MW} = 0.0017 \text{ MW}$

### 3. Data Center Infrastructure (PUE)
To account for cooling and power distribution, we apply the 

KeyboardInterrupt: Interrupted by user

In [10]:
chat_with_estimator()

Enter the path to your ML code directory:  /kaggle/input/datasets/katerynazakharova/ml-code/code/sample1



--- Starting Estimation Session (Type 'exit' to quit) ---


🤖 Gemma Estimator:
I have analyzed your code. You are performing a PEFT (Parameter-Efficient Fine-Tuning) task using LoRA on the **Gemma-2-27b-it** model, utilizing 4-bit quantization via `bitsandbytes` and the `SFTTrainer` from the TRL library.

The model size (27B parameters) is significant, and while 4-bit quantization reduces the memory footprint, the computational intensity during the backward pass on H100s will still drive high power draw.

To provide an accurate instantaneous power load estimate in Megawatts (MW), I need you to verify the following details:

1.  **GPU Count:** Exactly how many NVIDIA H100 SXM GPUs will you be using for this run? (Please specify a number between 1 and 10).
2.  **Dataset & Training Scale:**
    *   **Dataset Size:** Your config mentions `num_dataset_samples: 1000`. Is this the final size you intend to use?
    *   **Batch Size:** Your config shows `per_device_train_batch_size: 1` with `g

You (Verification/Details):  10 GPUs



🤖 Gemma Estimator:
Thank you for the verification. Based on your configuration and the use of 10 NVIDIA H100 SXM GPUs, here is the peak instantaneous power load estimation.

### 1. Hardware Power Calculation
*   **GPU Power:** 10 GPUs $\times$ 700W = **7,000W**
*   **Host System Overhead:** 
    *   The overhead is 1,000W per 4-GPU block.
    *   For 10 GPUs, we have $10 / 4 = 2.5$ blocks.
    *   $2.5 \times 1,000\text{W} = \mathbf{2,500\text{W}}$
*   **Total Hardware Power:** $7,000\text{W} + 2,500\text{W} = \mathbf{9,500\text{W}}$ (or $0.0095\text{ MW}$)

### 2. Data Center Infrastructure (PUE)
To account for cooling, power distribution, and lighting, we apply the Power Usage Effectiveness (PUE) multiplier:
*   **Total Facility Power:** $0.0095\text{ MW} \times 1.2\text{ PUE} = \mathbf{0.0114\text{ MW}}$

### Final Estimate
The estimated instantaneous peak power load required to run your Gemma-2-27b fine-tuning code on 10x H100 GPUs is:

**0.0114 MW**



KeyboardInterrupt: Interrupted by user